In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

STUDENT TASKS

In [0]:
#  TASK 1: Create Delta Table
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/my_volume/delta_table")

In [0]:
# TASK 2: Insert New Data
new_data = [(5, "C005", "Camera", 30000)]
new_df = spark.createDataFrame(new_data, columns)
new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/my_volume/delta_table")


In [0]:
# TASK 3: Update Data
# Update:
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark, "/Volumes/workspace/default/my_volume/delta_table")
delta_table.update(
    condition="id =2",
    set = {"amount":"180000"}
)

DataFrame[num_affected_rows: bigint]

In [0]:
#  Delete:
# id = 1
delta_table.delete("id = 1")


DataFrame[num_affected_rows: bigint]

TASK 5: MERGE (Incremental Load)

In [0]:
#  TASK 5: MERGE (Incremental Load)
#  New incoming data:

# updates = [
#     (3, "C003", "Tablet", 22000),  # update
#     (6, "C006", "Watch", 8000)     # insert
# ]
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()


id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "amount": "source.amount",
    "customer_id": "source.customer_id",
    "product": "source.product"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "customer_id": "source.customer_id",
    "product": "source.product",
    "amount": "source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

TASK 6: Schema Evolution

In [0]:
from pyspark.sql.functions import lit

df_updated = spark.read.format("delta").load("/Volumes/workspace/default/my_volume/delta_table") \
    .withColumn("country", lit("India"))

df_updated.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/my_volume/delta_table")

TASK 7: Time Travel

In [0]:
delta_table.history().show()

+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     11|2026-04-14 16:49:58|72387054655284|22pa1a0475@vishnu...|    WRITE|{mode -> Overwrit...|NULL|{1435853665351440}|   01d78191-0fc3-411...|0414-161251-u24nk...|         10|WriteSerializable|        fa

Previous version

In [0]:
old_df = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load("/Volumes/workspace/default/my_volume/delta_table")

old_df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  2|       C002| Mobile|180000|
+---+-----------+-------+------+



Restore old data

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]